In [15]:
import pandas as pd

# Read CSV
df = pd.read_csv("../data/customer_churn_train.csv")

# Drop the customer ID column as it has no useful information
df = df.drop(columns=["CustomerID", "Subscription Type"])

# Remove NaN
df = df[df["Churn"].notna()]

# Display head
df.head()

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn
0,30.0,Female,39.0,14.0,5.0,18.0,Annual,932.0,17.0,1.0
1,65.0,Female,49.0,1.0,10.0,8.0,Monthly,557.0,6.0,1.0
2,55.0,Female,14.0,4.0,6.0,18.0,Quarterly,185.0,3.0,1.0
3,58.0,Male,38.0,21.0,7.0,7.0,Monthly,396.0,29.0,1.0
4,23.0,Male,32.0,20.0,5.0,8.0,Monthly,617.0,20.0,1.0


In [16]:
# Encode Gender, Contract Length
df["Gender"] = df["Gender"].astype("category")
df["Contract Length"] = df["Contract Length"].astype("category")

In [17]:
# Split into X and y
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [29]:
# Split into train, validation and test
from sklearn.model_selection import train_test_split

# 1. First split: 10% for Test, 90% for Temp (Train + Val)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

# 2. Second split: 11.11% of Temp goes to Val (which is 10% of total)
# 8/9 of Temp stays in Train (which is 80% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)

# Verification
print(f"Train: {len(X_train) / len(X):.0%}")
print(f"Val:   {len(X_val) / len(X):.0%}")
print(f"Test:  {len(X_test) / len(X):.0%}")

Train: 80%
Val:   10%
Test:  10%


In [52]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import xgboost as xgb

# 1. Combine Train and Val for the search, but tell it which is which
# We create a list where -1 = training and 0 = validation
split_index = [-1] * len(X_train) + [0] * len(X_val)
pds = PredefinedSplit(test_fold=split_index)

X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

# 2. Define model and grid
base_model = xgb.XGBClassifier(enable_categorical=True, tree_method="hist", n_estimators=1000, learning_rate=0.1)

param_grid = {
    "max_depth": [1, 2, 3],
    "alpha": [1, 2],
    'colsample_bytree': [0.1, 0.5, 0.8]
}

# 3. Setup Search with cv=pds
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=pds,  # This tells it NOT to do cross-validation
    verbose=1,
)

# 4. Fit and Print
grid_search.fit(X_combined, y_combined)

print("--- Best Params (Single Split) ---")
print(grid_search.best_params_)
print(f"Best Score: {grid_search.best_score_:.4f}")

Fitting 1 folds for each of 18 candidates, totalling 18 fits
--- Best Params (Single Split) ---
{'alpha': 1, 'colsample_bytree': 0.5, 'max_depth': 2}
Best Score: 0.9990


In [54]:
# Calculate on the test set
test_accuracy = grid_search.score(X_test, y_test)
print(f"Final Test Accuracy: {test_accuracy:.2%}")

Final Test Accuracy: 99.92%


In [55]:
import pandas as pd

# 1. Get predictions
y_pred = grid_search.predict(X_test)

# 2. Create a DataFrame for comparison
# We convert X_test to a DataFrame if it isn't one already to see the features
error_df = X_test.copy()
if not isinstance(error_df, pd.DataFrame):
    error_df = pd.DataFrame(error_df)

# 3. Add the Actual and Predicted columns
error_df["Actual"] = y_test.values if hasattr(y_test, "values") else y_test
error_df["Predicted"] = y_pred

# 4. Filter for rows where they don't match
errors = error_df[error_df["Actual"] != error_df["Predicted"]]

print(f"Total Errors: {len(errors)}")
display(errors.head(n=16))

Total Errors: 34


,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Actual,Predicted
42455,48.0,Male,5.0,4.0,1.0,17.0,Quarterly,793.00,17.0,1.0,0
761,22.0,Male,1.0,24.0,1.0,18.0,Annual,638.00,14.0,1.0,0
64337,38.0,Female,3.0,24.0,0.0,3.0,Annual,765.00,11.0,1.0,0
101263,48.0,Male,44.0,25.0,3.0,20.0,Annual,498.00,3.0,1.0,0
9485,33.0,Male,39.0,24.0,3.0,4.0,Annual,498.00,27.0,1.0,0
27487,33.0,Female,5.0,11.0,2.0,11.0,Quarterly,855.00,11.0,1.0,0
237882,38.0,Male,31.0,22.0,0.0,9.0,Annual,498.43,6.0,1.0,0
19022,28.0,Male,57.0,29.0,1.0,15.0,Quarterly,498.00,9.0,1.0,0
29353,22.0,Female,1.0,17.0,1.0,18.0,Quarterly,550.00,4.0,1.0,0
177954,39.0,Male,3.0,27.0,2.0,16.0,Quarterly,546.00,4.0,1.0,0


In [57]:
# Test on the test set
test_df = pd.read_csv("../data/customer_churn_test.csv")

# Drop the customer ID column as it has no useful information
test_df = test_df.drop(columns=["CustomerID", "Subscription Type"])

# Make into category
test_df["Gender"] = test_df["Gender"].astype("category")
test_df["Contract Length"] = test_df["Contract Length"].astype("category")

# Split into X and y
X_test_true, y_test_true = test_df.drop(columns=["Churn"]), test_df["Churn"]

# Calculate accuracy
test_accuracy = grid_search.score(X_test_true, y_test_true)
print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.5039
